<a href="https://colab.research.google.com/github/ojohnso3-oss/ojohnso3-INST414/blob/main/Lab12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import json

import pandas as pd
import numpy as np

from scipy.sparse import lil_matrix

import matplotlib.pyplot as plt


In [3]:
actor_name_map = {}
movie_actor_map = {}
actor_genre_map = {}


with open("/content/sample_data/imdb_movies_2000to2022.prolific.json", "r") as in_file:
    for line in in_file:

        # Read the movie on this line and parse its json
        this_movie = json.loads(line)

        # Skip movies with no ratings
        if len(this_movie["rating"]) == 0:
            continue

        # Add all actors to the id->name map
        for actor_id,actor_name in this_movie['actors']:
            actor_name_map[actor_id] = actor_name

        # For each actor, add this movie's genres to that actor's list
        for actor_id,actor_name in this_movie['actors']:
            this_actors_genres = actor_genre_map.get(actor_id, {})

            # Increment the count of genres for this actor
            for g in this_movie["genres"]:
                this_actors_genres[g] = this_actors_genres.get(g, 0) + 1

            # Update the map
            actor_genre_map[actor_id] = this_actors_genres

        # Finished with this film
        movie_actor_map[this_movie["imdb_id"]] = ({
            "movie": this_movie["title"],
            "actors": set([item[0] for item in this_movie['actors']]),
            "genres": this_movie["genres"],
            "rating": this_movie["rating"]["avg"]
        })

In [4]:
actor_id_to_index = {}
for i, actor_id in enumerate(actor_name_map.keys()):
    actor_id_to_index[actor_id] = i
n_actors = len(actor_name_map)
n_actors
movie_ids = list(movie_actor_map.keys())
n_movies = len(movie_ids)

In [7]:
from scipy.sparse import lil_matrix, csr_matrix
X = lil_matrix((n_movies, n_actors), dtype = np.int8)
for row, movie_id in enumerate(movie_ids):
    for actor_id in movie_actor_map[movie_id]["actors"]:
        col = actor_id_to_index[actor_id]
        X[row, col] = 1
X = csr_matrix(X)
y = np.array([movie_actor_map[mid]["genres"][0] for mid in movie_ids])
y_full = [movie_actor_map[mid] for mid in movie_ids]
y = np.array([movie_actor_map[mid]["genres"][0] for mid in movie_ids])
y_full = [movie_actor_map[mid] for mid in movie_ids]

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, y_full_train, y_full_test = train_test_split(X, y, y_full, test_size = 0.2)

In [9]:
from sklearn.tree import DecisionTreeClassifier
##Depth = 1

depth = 1
clf = DecisionTreeClassifier(max_depth = depth)
clf.fit(X_train, y_train)
train_preds = clf.predict(X_train)
train_preds
y_train
correct = 0
for i, pred in enumerate(train_preds):
    if pred in y_train[i]:
        correct += 1
train_acc = correct /len(y_train)
train_acc
test_preds = clf.predict(X_test)

correct = 0
for i, pred in enumerate(test_preds):
    if pred in y_train[i]:
        correct += 1
test_acc = correct /len(y_test)
test_acc

In [11]:
for depth in range(1, 11):
    clf = DecisionTreeClassifier(max_depth = depth)
    clf.fit(X_train, y_train)

    train_preds = clf.predict(X_train)
    train_acc = np.mean([p in y_full_train[i]["genres"] for i, p in enumerate(train_preds)])

    test_preds = clf.predict(X_test)
    test_acc = np.mean([p in y_full_test[i]["genres"] for i, p in enumerate(test_preds)])

    print(f"Depth: {depth}, Train Acc: {train_acc}, Test Acc: {test_acc}")

Depth: 1, Train Acc: 0.3085854564755839, Test Acc: 0.29609976120986997
Depth: 2, Train Acc: 0.31024416135881105, Test Acc: 0.2982223401432741
Depth: 3, Train Acc: 0.3120355626326964, Test Acc: 0.29981427434332714
Depth: 4, Train Acc: 0.3134952229299363, Test Acc: 0.3016715309100557
Depth: 5, Train Acc: 0.31475583864118895, Test Acc: 0.30246749801008227
Depth: 6, Train Acc: 0.3162154989384289, Test Acc: 0.3029981427434333
Depth: 7, Train Acc: 0.3174097664543524, Test Acc: 0.30459007694348633
Depth: 8, Train Acc: 0.31833864118895966, Test Acc: 0.30565136641018836
Depth: 9, Train Acc: 0.3185376857749469, Test Acc: 0.30565136641018836
Depth: 10, Train Acc: 0.31993099787685775, Test Acc: 0.3059166887768639


I would say the relationship between depth and accuracy is xxxxxx